# CLOVERS: causal mediation through a renal and a pulmonary pathway

Second part of the analysis. Two 24-hour mediators of the effect of the fluid strategy on being alive and discharged by day 28: the change in serum creatinine from baseline to 24 hours (renal pathway) and escalation of respiratory support within 24 hours (pulmonary pathway).

The notebook builds the analysis cohorts from the BioLINCC files, describes the mediators, then writes and runs the CMAverse g-formula script (14 models, 1,000 bootstrap resamples, 10 imputations). Analyses are restricted to the 1,504 patients alive at 24 hours (landmark cohort). The pulmonary and joint models use the 1,408 of them not on invasive ventilation at randomisation (escalation cohort, called headroom in the code).

The adjustment set has eleven baseline characteristics (age, sex, SOFA score, lactate, creatinine, SpO2/FiO2, mean arterial pressure, chronic kidney disease, pulmonary source of sepsis, vasopressor at baseline, pre-randomisation fluid volume), with non-invasive respiratory support at baseline added in the escalation cohort.

With a logistic outcome model, the ratio effects that CMAverse returns (Rpnde, Rtnie and the others) are odds ratios of the counterfactual probabilities. The paper reports risk ratios, computed from the same bootstrap draws by `cmaverse_models.R` and `effect_scales.R`.

# Part A. Setup, data and landmark

## A1. R and CMAverse

In [1]:
import os
os.environ['R_HOME'] = r"C:\Program Files\R\R-4.4.3"
os.environ['PATH']  = r"C:\Program Files\R\R-4.4.3\bin\x64" + os.pathsep + os.environ['PATH']
os.environ['PYTHONIOENCODING'] = 'utf-8'

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats

# rpy2: separate console writers so that R's UTF-8 output cannot crash the kernel
import rpy2.rinterface_lib.callbacks as cb

_orig_consolewrite_print = cb.consolewrite_print
_orig_consolewrite_warnerror = cb.consolewrite_warnerror

def _safe_write(s):
    try:
        _orig_consolewrite_print(s)
    except Exception:
        print(s.encode('utf-8', errors='replace').decode('utf-8'))

def _safe_write_err(s):
    try:
        _orig_consolewrite_warnerror(s)
    except Exception:
        print(s.encode('utf-8', errors='replace').decode('utf-8'))

cb.consolewrite_print = _safe_write
cb.consolewrite_warnerror = _safe_write_err

%load_ext rpy2.ipython

In [2]:
%%R
# CMAverse + dependencies. If CMAverse must be compiled from source, Rtools 4.4 is
# required: https://cran.r-project.org/bin/windows/Rtools/
pkgs <- c("devtools", "jsonlite", "mice", "MASS", "lme4", "nnet", "ggplot2")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE))
    install.packages(p, repos = "https://cloud.r-project.org")
}
if (!requireNamespace("CMAverse", quietly = TRUE)) {
  message("Installing CMAverse from GitHub - requires Rtools 4.4")
  devtools::install_github("BS1125/CMAverse", upgrade = "never")
}
library(CMAverse)
library(jsonlite)
cat(sprintf("CMAverse %s ready\n", packageVersion("CMAverse")))

CMAverse 0.1.0 ready


## A2. Configuration

Settings of the run reported in the paper. `nRep` is passed to CMAverse but its g-formula estimator does not use it.

In [3]:
# Configuration. The two BioLINCC files are expected in data/.
dataset_path = "data/DATASET.csv"
derived_path = "data/DERIVED.csv"

OUTPUT_DIR = os.path.join(os.path.dirname(dataset_path), 'mediation')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Settings of the run reported in the paper
SEED = 42
N_BOOTSTRAP, N_MC_REP, N_MICE = 1000, 100, 10

# Mediators and outcomes
M_RENAL = 'delta_creat_0_24h'        # continuous  -> mreg gaussian/linear
# Pulmonary mediator: upward move on the 3-level respiratory scale (0 none,
#   1 HFNC/NIPPV/CPAP, 2 IMV) within 0-24 h, in patients at level 0 or 1 at baseline.
#   Level 0 escalates on any advanced support, level 1 only on IMV. Level 2
#   (IMV at baseline, home invasive ventilation included) is excluded.
#   See B4 and C2.
M_PULM  = 'resp_escalation_24h'      # binary      -> mreg logistic
PULM_COVAR = 'on_intermediate_baseline'   # baseline level 1, added to basec in the escalation-cohort models

# Alternative pulmonary mediators (Results E2)
M_PULM_COMPOSITE = 'new_adv_support_24h'  # new advanced support, baseline level 0 only
M_PULM_SENS_SF   = 'delta_sf_ratio_0_24h' # change in SpO2/FiO2
M_PULM_NESTED    = 'new_imv_24h'          # new invasive ventilation

PRIMARY_OUTCOME   = 'alive_discharged_28d'
SECONDARY_OUTCOME = 'mortality_90d'

print(f"Output directory : {OUTPUT_DIR}")
print(f"Settings         : nboot={N_BOOTSTRAP}, nRep={N_MC_REP}, mice m={N_MICE}")
print( "Estimator        : joint two-mediator g-formula, natural-effects (CMAverse gformula)")
print(f"Mediators        : renal={M_RENAL} (linear) | pulmonary={M_PULM} (logistic, gradient escalation)")
print(f"Pulmonary covar. : {PULM_COVAR} (baseline intermediate support; headroom-cohort models only)")
print(f"Pulmonary alt.   : {M_PULM_COMPOSITE} (de-novo, level-0) | {M_PULM_NESTED} (IMV-only) | {M_PULM_SENS_SF} (ceiling)")
print(f"Outcome          : primary={PRIMARY_OUTCOME} | secondary={SECONDARY_OUTCOME}")
print( "postc (L1)       : EMPTY  (fluid/vaso/RRT are mechanism, not confounders)")
print( "Escalation cohort: headroom (landmark survivors at baseline respiratory level 0 or 1)")


Output directory : data\mediation
Settings         : nboot=1000, nRep=100, mice m=10
Estimator        : joint two-mediator g-formula, natural-effects (CMAverse gformula)
Mediators        : renal=delta_creat_0_24h (linear) | pulmonary=resp_escalation_24h (logistic, gradient escalation)
Pulmonary covar. : on_intermediate_baseline (baseline intermediate support; headroom-cohort models only)
Pulmonary alt.   : new_adv_support_24h (de-novo, level-0) | new_imv_24h (IMV-only) | delta_sf_ratio_0_24h (ceiling)
Outcome          : primary=alive_discharged_28d | secondary=mortality_90d
postc (L1)       : EMPTY  (fluid/vaso/RRT are mechanism, not confounders)
Escalation cohort: headroom (landmark survivors at baseline respiratory level 0 or 1)


## A3. Data loading

In [4]:
# Load raw + derived data
df_raw     = pd.read_csv(dataset_path, low_memory=False)
df_derived = pd.read_csv(derived_path, low_memory=False)

ids_withdrawn = df_raw.loc[df_raw['term_wd'] == 'Yes', 'id'].unique()
df_raw     = df_raw[~df_raw['id'].isin(ids_withdrawn)]
df_derived = df_derived[~df_derived['id'].isin(ids_withdrawn)]
print(f"Total patients after excluding {len(ids_withdrawn)} withdrawn: {df_raw['id'].nunique()}")

screening = df_raw[df_raw['intervalname'] == 'Screening'].copy()
baseline  = df_raw[df_raw['intervalname'] == 'Baseline'].copy()
sip       = df_raw[df_raw['intervalname'] == 'Study intervention period'].copy()
day1      = df_raw[df_raw['intervalname'] == 'Day 1'].copy()

derived_baseline = df_derived[df_derived['intervalname'] == 'Baseline'].copy()
derived_sip      = df_derived[df_derived['intervalname'] == 'Study intervention period'].copy()

all_patient_ids = baseline['id'].unique()
print(f"Baseline patients: {len(all_patient_ids)}")
print(f"Day 1 patients:    {day1['id'].nunique()}")

Total patients after excluding 23 withdrawn: 1540
Baseline patients: 1540
Day 1 patients:    1485


## Values for Table 1 and effect of the strategy on each mediator

These two cells print the values of Table 1 next to the expected ones. The second reads the cohort files written in D1, so it is run after D1.

In [6]:
# Values for Table 1: fluid volume, 24-hour changes and respiratory support at baseline.
# Needs df_raw and df_derived (A3). Same definitions as B2 and B4, inside a function
# so that no notebook variable is overwritten.
def _tables_1_2():
    import numpy as np
    import pandas as pd
    from scipy.stats import mannwhitneyu, fisher_exact

    def yes(s):                                   # same rule as _is_yes (B2)
        if pd.api.types.is_numeric_dtype(s):
            return s.fillna(0) > 0
        return s.astype(str).str.strip().str.lower().isin(['yes', '1', 'true', 'y'])

    wd = df_raw.loc[df_raw['term_wd'] == 'Yes', 'id'].unique()
    d = df_raw[~df_raw['id'].isin(wd)]
    bl = d[d['intervalname'] == 'Baseline'].groupby('id').first()
    d1 = d[d['intervalname'] == 'Day 1'].groupby('id').first()
    sip = d[d['intervalname'] == 'Study intervention period']
    ids = bl.index
    num = lambda df, c: pd.to_numeric(df[c], errors='coerce').reindex(ids)

    def flag(c):                                  # same rule as sip_flag (B2)
        return (sip.assign(_y=yes(sip[c])).groupby('id')['_y'].max()
                .reindex(ids).fillna(False).astype(bool))

    R = bl['rand_trt'].str.contains('Restrictive', case=False, na=False)
    L = ~R
    lm = ~(num(bl, 'term_deathdt') <= 1)
    ivb = flag('ipvent_iventb')
    home = bl['medhx_homemechvent'] == 'Yes'
    imv_b = ivb | home
    int_b = (flag('ipvent_hfo2b') | flag('ipvent_cpapb')) & ~imv_b
    lvl = pd.Series(np.where(imv_b, 2, np.where(int_b, 1, 0)), index=ids)
    imv_w = flag('ipvent_iventh6') | flag('ipvent_iventh24')
    int_w = flag('ipvent_hfo2h6') | flag('ipvent_hfo2h24') | flag('ipvent_cpaph6') | flag('ipvent_cpaph24')
    peak = pd.Series(np.maximum(lvl.values, np.where(imv_w, 2, np.where(int_w, 1, 0))), index=ids)
    esc = (peak > lvl).astype(float).where(lvl <= 1)
    ec = lm & (lvl <= 1)
    print(f"CHECK  analysis {len(ids)} (1540) | landmark {int(lm.sum())} (1504) | "
          f"invasive at baseline in landmark {int((lm & (lvl == 2)).sum())} (96) | escalation {int(ec.sum())} (1408)")

    def med(x, k=0):
        x = x.dropna()
        return f"{x.median():,.{k}f} [{x.quantile(.25):,.{k}f} to {x.quantile(.75):,.{k}f}] (n={len(x)})"
    def nN(s):
        s = s.dropna()
        return f"{int(s.sum())}/{len(s)} ({100 * s.mean():.1f})"
    def fp(p):
        return '<0.001' if p < 0.001 else f'{p:.3f}'
    def fisher(s):
        r, l = s[R].dropna(), s[L].dropna()
        return fisher_exact([[int(r.sum()), int(len(r) - r.sum())],
                             [int(l.sum()), int(len(l) - l.sum())]])[1]
    def mw(s):
        return mannwhitneyu(s[R].dropna(), s[L].dropna(), alternative='two-sided').pvalue

    # 1. Fluid volume over the protocol period (d_ipfluid_all, DERIVED)
    fl = pd.to_numeric(df_derived[df_derived['intervalname'] == 'Baseline']
                       .groupby('id')['d_ipfluid_all'].first(), errors='coerce').reindex(ids)
    print(f"1. Fluid volume 0-24 h, mL | R {med(fl[R])} | L {med(fl[L])} | missing {int(fl.isna().sum())} (4) "
          f"| Mann-Whitney p {fp(mw(fl))}   [expected R 1,280 [573 to 2,290], L 3,417 [2,503 to 4,499]]")

    # 3. Changes over 0-24 h in the landmark cohort (first day-1 value minus baseline)
    sf = lambda df: (num(df, 'labs_spo2') / num(df, 'labs_spo2fio2')).replace([np.inf, -np.inf], np.nan)
    dsf = (sf(d1) - sf(bl)).where(lm)
    dcr = (num(d1, 'labs_creatinine') - num(bl, 'labs_creatinine')).where(lm)
    wor = (dcr > 0).astype(float).where(dcr.notna())
    for lab, s, k, att in [
            ('Delta creatinine, mg/dL', dcr, 2, 'R -0.14 [-0.39 to 0.00], L -0.21 [-0.50 to -0.06], missing 126, p <0.001'),
            ('Delta SpO2/FiO2', dsf, 1, 'R 0.0 [-14.3 to 19.1], L 0.0 [-19.1 to 17.4], missing not given, p 0.086')]:
        m = s.isna() & lm
        print(f"3. {lab} | R {med(s[R], k)} | L {med(s[L], k)} | missing in landmark {int(m.sum())} "
              f"(R {int((m & R).sum())}, L {int((m & L).sum())}) | Mann-Whitney p {fp(mw(s))}   [expected {att}]")
    print(f"   Creatinine increase | R {nN(wor[R])} | L {nN(wor[L])} | Fisher p {fp(fisher(wor))}"
          f"   [expected 163/686, 116/692, 0.001]")

    # 4. Respiratory support at baseline by arm
    noms = {0: 'room air or standard O2', 1: 'non-invasive support', 2: 'invasive ventilation'}
    for titre, msk, att in [("analysis cohort", pd.Series(True, index=ids), '1333 / 97 / 110'),
                            ('landmark cohort', lm, '1317 / 91 / 96')]:
        print(f"4. Support at baseline, {titre} (expected totals {att})")
        for v in (0, 1, 2):
            nr, nl = int(((lvl == v) & msk & R).sum()), int(((lvl == v) & msk & L).sum())
            print(f"   {noms[v]:<28} R {nr} ({100 * nr / int((msk & R).sum()):.1f}) | "
                  f"L {nl} ({100 * nl / int((msk & L).sum()):.1f}) | total {nr + nl}")

    # New invasive ventilation (Table 1)
    s = imv_w.astype(float).where(~ivb)
    print(f"   New invasive ventilation, excluding ipvent_iventb | R {nN(s[R])} | L {nN(s[L])} | Fisher p {fp(fisher(s))}")
    print("   [expected 45/717, 49/714, 0.671]")
    print(f"   Escalation, escalation cohort | R {nN(esc.where(ec)[R])} | L {nN(esc.where(ec)[L])} | "
          f"Fisher p {fp(fisher(esc.where(ec)))}   [expected 66/703, 81/705, 0.222]")

_tables_1_2()

CHECK  analysis 1540 (1540) | landmark 1504 (1504) | invasive at baseline in landmark 96 (96) | escalation 1408 (1408)
1. Fluid volume 0-24 h, mL | R 1,280 [573 to 2,290] (n=766) | L 3,417 [2,503 to 4,499] (n=770) | missing 4 (4) | Mann-Whitney p <0.001   [expected R 1,280 [573 to 2,290], L 3,417 [2,503 to 4,499]]
3. Delta creatinine, mg/dL | R -0.14 [-0.39 to 0.00] (n=686) | L -0.21 [-0.50 to -0.06] (n=692) | missing in landmark 126 (R 62, L 64) | Mann-Whitney p <0.001   [expected R -0.14 [-0.39 to 0.00], L -0.21 [-0.50 to -0.06], missing 126, p <0.001]
3. Delta SpO2/FiO2 | R 0.0 [-14.3 to 19.0] (n=677) | L 0.0 [-19.0 to 17.4] (n=700) | missing in landmark 127 (R 71, L 56) | Mann-Whitney p 0.086   [expected R 0.0 [-14.3 to 19.1], L 0.0 [-19.1 to 17.4], missing not given, p 0.086]
   Creatinine increase | R 163/686 (23.8) | L 116/692 (16.8) | Fisher p 0.001   [expected 163/686, 116/692, 0.001]
4. Support at baseline, analysis cohort (expected totals 1333 / 97 / 110)
   room air or stan

In [7]:
%%R -i OUTPUT_DIR
## Adjusted effect of the strategy on each mediator (Results, treatment separation).
## run_cmest does not keep the cmest objects, and one set.seed(42) precedes all the
## models, so the CMAverse mediator regression is refitted here (same formula, same
## mice call on the exported cohort, m = 10), combined by Rubin's rules.
suppressPackageStartupMessages(library(mice))
local({
  BASEC <- c("age", "female", "sofa_baseline", "lactate_baseline", "creatinine_baseline",
             "sf_ratio_baseline", "map_baseline", "ckd_any", "sepsis_source_pulmonary",
             "on_vasopressor_baseline", "prerand_fluid_L")
  run <- function(csv, med, covs, fam, lab) {
    df <- read.csv(file.path(OUTPUT_DIR, csv), stringsAsFactors = FALSE)
    df$restrictive <- as.integer(df$restrictive)
    covs <- covs[sapply(covs, function(v) length(unique(na.omit(df[[v]]))) > 1)]
    set.seed(42)
    imp <- suppressWarnings(mice(df, m = 10, print = FALSE))
    f <- as.formula(paste(med, "~", paste(c("restrictive", covs), collapse = " + ")))
    fits <- lapply(seq_len(imp$m), function(i) glm(f, family = fam, data = complete(imp, i)))
    s <- summary(pool(as.mira(fits)), conf.int = TRUE)
    r <- s[as.character(s$term) == "restrictive", ]
    est <- c(r$estimate, r[["2.5 %"]], r[["97.5 %"]])
    logit <- fam$family == "binomial"
    if (logit) est <- exp(est)
    cat(sprintf("%s | n %d | covariates %d | %s %.3f (%.3f to %.3f), p %s | missing mediator %d\n",
                lab, nrow(df), length(covs), if (logit) "odds ratio" else "difference mg/dL",
                est[1], est[2], est[3], if (r$p.value < 0.001) "<0.001" else sprintf("%.3f", r$p.value),
                sum(is.na(df[[med]]))))
  }
  run("cohort_landmark.csv", "delta_creat_0_24h", BASEC, gaussian(),
      "2. Creatinine, landmark cohort")
  run("cohort_headroom.csv", "resp_escalation_24h", c(BASEC, "on_intermediate_baseline"), binomial(),
      "5. Escalation, escalation cohort")
})

2. Creatinine, landmark cohort | n 1504 | covariates 11 | difference mg/dL 0.096 (0.030 to 0.162), p 0.005 | missing mediator 126
5. Escalation, escalation cohort | n 1408 | covariates 12 | odds ratio 0.801 (0.554 to 1.158), p 0.238 | missing mediator 0


## A4. Early deaths and the 24-hour landmark

Patients who died within 24 hours have no 24-hour mediator value and are excluded. The cell compares early deaths between arms.

In [5]:
# Early deaths (<24 h) by arm
_arm = baseline.groupby('id')['rand_trt'].first()
_restrictive = _arm.str.contains('Restrictive', case=False, na=False).astype(int)

_death_day = pd.to_numeric(baseline.groupby('id')['term_deathdt'].first(), errors='coerce')
_alive_24h = pd.Series(1, index=_restrictive.index, dtype=int)
_alive_24h[_death_day.notna() & (_death_day <= 1)] = 0

_tab = pd.DataFrame({'restrictive': _restrictive, 'alive_at_24h': _alive_24h})
early = _tab.groupby('restrictive').agg(n_total=('alive_at_24h', 'count'),
                                        n_alive=('alive_at_24h', 'sum'))
early['n_died']   = early['n_total'] - early['n_alive']
early['pct_died'] = (early['n_died'] / early['n_total'] * 100).round(2)
early.index = early.index.map({0: 'Liberal', 1: 'Restrictive'})

print("EARLY DEATHS (<24h) BY ARM"); print("=" * 60)
print(early.to_string())

ct = pd.crosstab(_tab['restrictive'], _tab['alive_at_24h'])
if ct.shape == (2, 2):
    _, p_ed = stats.fisher_exact(ct)
    print(f"\nFisher exact (early-death imbalance): p = {p_ed:.3f}")

print(f"\nDied <24h (excluded): {(_tab['alive_at_24h'] == 0).sum()}")
print(f"24h-landmark cohort : {(_tab['alive_at_24h'] == 1).sum()}")

EARLY DEATHS (<24h) BY ARM
             n_total  n_alive  n_died  pct_died
restrictive                                    
Liberal          770      756      14      1.82
Restrictive      770      748      22      2.86

Fisher exact (early-death imbalance): p = 0.237

Died <24h (excluded): 36
24h-landmark cohort : 1504


# Part B. Variables

## B1. Exposure

In [6]:
# A: treatment assignment (CRF: "Liberal Fluid Group" / "Restrictive Fluid Group")
#   restrictive = 1, liberal = 0  ->  astar=0 (liberal), a=1 (restrictive)
A = baseline.groupby('id')['rand_trt'].first().to_frame()
A['restrictive'] = A['rand_trt'].str.contains('Restrictive', case=False, na=False).astype(int)

print("Treatment assignment:")
print(A['restrictive'].value_counts().rename({0: 'Liberal', 1: 'Restrictive'}))
assert A['restrictive'].nunique() == 2, f"ERROR: {A['rand_trt'].unique()}"
print("OK")

Treatment assignment:
restrictive
Restrictive    770
Liberal        770
Name: count, dtype: int64
OK


## B2. Baseline confounders and respiratory support at baseline

The eleven confounders, and the respiratory level at baseline: 0 none, 1 non-invasive support (high-flow oxygen, NIV or CPAP), 2 invasive ventilation, home invasive ventilation included. The level defines the cohorts, and the indicator of level 1 is added to the adjustment set in the escalation cohort.

In [7]:
# L0: baseline confounders, raw-flag helper and the 3-level respiratory scale

# raw ventilation-flag reducer (SIP rows carry the ipvent_* flags)
def _is_yes(s):
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0) > 0
    return s.astype(str).str.strip().str.lower().isin(['yes', '1', 'true', 'y'])

def sip_flag(col):
    # Per-patient max(is_yes) over Study-intervention-period rows; reindexed to all
    # patients, missing -> False. Returns a boolean Series indexed by id.
    idx = pd.Index(all_patient_ids, name='id')
    if col not in sip.columns:
        print(f"  WARNING: {col} not in SIP columns -> all False")
        return pd.Series(False, index=idx)
    tmp = sip.assign(_y=_is_yes(sip[col])).groupby('id')['_y'].max()
    return tmp.reindex(idx).fillna(False).astype(bool)

_idx = pd.Index(all_patient_ids, name='id')

# baseline respiratory-support flags (t=b)
on_hfnc_baseline    = sip_flag('ipvent_hfo2b')
on_cpap_baseline    = sip_flag('ipvent_cpapb')
on_imv_baseline_raw = sip_flag('ipvent_iventb')                     # acute invasive at baseline
# Home INVASIVE ventilation (medhx_homemechvent; per the data dictionary this flag EXCLUDES
# NIV). Folded into the baseline-invasive flag so the single chronic-vent patient not captured
# by ipvent_iventb (crosstab: 12 home-invasive, 11 already flagged, 1 unflagged) is placed at
# gradient level 2 and cannot be miscounted as a new escalation event.
home_imv = (baseline.groupby('id')['medhx_homemechvent'].first() == 'Yes').reindex(_idx).fillna(False)
on_imv_baseline = (on_imv_baseline_raw | home_imv)

# 3-level respiratory gradient at baseline
#   level 0 = no advanced support | level 1 = intermediate (HFNC/NIPPV/CPAP) | level 2 = IMV
on_int_baseline = (on_hfnc_baseline | on_cpap_baseline) & ~on_imv_baseline
base_resp_lvl = pd.Series(np.where(on_imv_baseline, 2, np.where(on_int_baseline, 1, 0)), index=_idx)
on_intermediate_baseline = (base_resp_lvl == 1).astype(int)        # covariate of the escalation-cohort models
on_adv_support_baseline  = (base_resp_lvl >= 1).astype(int)        # any advanced support (incl. home-invasive)

# Demographics
demographics = screening.groupby('id')['scr_age'].first().to_frame('age')
demographics['female'] = (screening.groupby('id')['dmg_sex'].first() == 'Female').astype(int)

# Baseline labs (SF = SpO2 / FiO2; labs_spo2fio2 holds FiO2)
labs_baseline = baseline.groupby('id').agg({
    'labs_lactate':    'first', 'labs_creatinine': 'first',
    'labs_spo2':       'first', 'labs_spo2fio2':   'first',
}).rename(columns={
    'labs_lactate': 'lactate_baseline', 'labs_creatinine': 'creatinine_baseline',
    'labs_spo2': 'spo2_baseline', 'labs_spo2fio2': 'fio2_baseline',
})
labs_baseline['sf_ratio_baseline'] = (
    labs_baseline['spo2_baseline'] / labs_baseline['fio2_baseline']
).replace([np.inf, -np.inf], np.nan)

# MAP
vitals = baseline.groupby('id').agg({'bp_systolic': 'first', 'bp_diastolic': 'first'})
vitals['map_baseline'] = vitals['bp_diastolic'] + (vitals['bp_systolic'] - vitals['bp_diastolic']) / 3

# SOFA (no GCS component)
sofa_data = baseline.groupby('id').agg({
    'labs_platelets': 'first', 'labs_totalbili': 'first', 'labs_creatinine': 'first',
    'bp_systolic': 'first', 'bp_diastolic': 'first', 'blvaso_yn': 'first',
    'labs_pao2': 'first', 'labs_pao2fio2': 'first', 'medhx_cdialysisyn': 'first',
})
sofa = pd.DataFrame(index=sofa_data.index)
sofa['sofa_coag'] = 0
for t, s in [(150,1),(100,2),(50,3),(20,4)]:
    sofa.loc[sofa_data['labs_platelets'].notna() & (sofa_data['labs_platelets'] <= t), 'sofa_coag'] = s
sofa['sofa_liver'] = 0
for t, s in [(1.2,1),(2.0,2),(6.0,3),(12.0,4)]:
    sofa.loc[sofa_data['labs_totalbili'].notna() & (sofa_data['labs_totalbili'] >= t), 'sofa_liver'] = s
dialysis = sofa_data['medhx_cdialysisyn'] == 'Yes'
sofa['sofa_renal'] = 0.0
for t, s in [(1.2,1),(2.0,2),(3.5,3),(5.0,4)]:
    sofa.loc[~dialysis & sofa_data['labs_creatinine'].notna() & (sofa_data['labs_creatinine'] >= t), 'sofa_renal'] = s
sofa.loc[dialysis, 'sofa_renal'] = np.nan
map_v = sofa_data['bp_diastolic'] + (sofa_data['bp_systolic'] - sofa_data['bp_diastolic']) / 3
sofa['sofa_cardio'] = 0
sofa.loc[map_v.notna() & (map_v < 70), 'sofa_cardio'] = 1
sofa.loc[sofa_data['blvaso_yn'] == 'Yes', 'sofa_cardio'] = 3
pf = np.where(sofa_data['labs_pao2fio2'].notna() & (sofa_data['labs_pao2fio2'] > 0),
              sofa_data['labs_pao2'] / sofa_data['labs_pao2fio2'], np.nan)
sofa['sofa_resp'] = 0
for t, s in [(400,1),(300,2),(200,3),(100,4)]:
    sofa.loc[pd.notna(pf) & (pf <= t), 'sofa_resp'] = s
sofa['sofa_baseline'] = sofa[['sofa_coag','sofa_liver','sofa_renal','sofa_cardio','sofa_resp']].sum(axis=1, min_count=1)

# Kidney comorbidity
# charl_kidney is text ('Present', 'Present with dialysis', 'Not present'). It is combined
# with baseline dialysis into one CKD-ever flag (no missing value).
_ck = baseline.groupby('id')['charl_kidney'].first()
_cd = baseline.groupby('id')['medhx_cdialysisyn'].first()
ckd_any = (_ck.isin(['Present', 'Present with dialysis']) | (_cd == 'Yes')).astype(int)

sepsis_pulmonary = baseline.groupby('id')['infrev_primsrc'].first().str.contains(
    'Pneumonia|Lung|lung|pneumonia|Pulmonary|pulmonary', na=False).astype(int)
on_vaso_bl = (baseline.groupby('id')['blvaso_yn'].first() == 'Yes').astype(int)
on_mv_bl   = (baseline.groupby('id')['venthx_randint'].first() == 'Yes').astype(int)  # IMV at baseline
prerand_fluid_L = derived_baseline.groupby('id')['d_prerand_fltotal'].first() / 1000
chronic_dialysis = (baseline.groupby('id')['medhx_cdialysisyn'].first() == 'Yes').astype(int) \
                     .reindex(all_patient_ids, fill_value=0)

# Assemble L0
L0 = demographics.copy()
L0['lactate_baseline']          = labs_baseline['lactate_baseline']
L0['creatinine_baseline']       = labs_baseline['creatinine_baseline']
L0['sf_ratio_baseline']         = labs_baseline['sf_ratio_baseline']
L0['map_baseline']              = vitals['map_baseline']
L0['sofa_baseline']             = sofa['sofa_baseline']
L0['ckd_any']                   = ckd_any
L0['sepsis_source_pulmonary']   = sepsis_pulmonary
L0['on_vasopressor_baseline']   = on_vaso_bl
L0['on_mech_vent_baseline']     = on_mv_bl
L0['prerand_fluid_L']           = prerand_fluid_L
L0['chronic_dialysis']          = chronic_dialysis
# variables that define the cohorts, and the escalation-cohort covariate (not in BASEC):
L0['on_adv_support_baseline']   = on_adv_support_baseline.reindex(L0.index).fillna(0).astype(int)
L0['base_resp_lvl']             = base_resp_lvl.reindex(L0.index).fillna(0).astype(int)
L0['on_intermediate_baseline']  = on_intermediate_baseline.reindex(L0.index).fillna(0).astype(int)

# basec = the 11 confounders passed to cmest for the renal models (landmark cohort).
#   ckd_any (CKD-ever) includes chronic dialysis, and chronic_dialysis is kept as a column
#   only for the cohort without chronic dialysis.
#   The escalation-cohort models add on_intermediate_baseline to BASEC
#   (assembled as BASEC_PULM in the R script), because baseline respiratory level is a
#   confounder there.
BASEC = ['age', 'female', 'sofa_baseline', 'lactate_baseline', 'creatinine_baseline',
         'sf_ratio_baseline', 'map_baseline', 'ckd_any', 'sepsis_source_pulmonary',
         'on_vasopressor_baseline', 'prerand_fluid_L']

print(f"L0: {len(L0)} patients x {len(L0.columns)} variables")
print(f"home-invasive vent (medhx_homemechvent): {int(home_imv.sum())}  "
      f"(folded into baseline IMV; {int((home_imv & ~on_imv_baseline_raw).sum())} not otherwise flagged)")
print("Baseline respiratory gradient (all randomised):")
for lvl, lab in [(0,'none'), (1,'intermediate HFNC/NIV/CPAP'), (2,'IMV (incl. home-invasive)')]:
    k = int((base_resp_lvl == lvl).sum())
    print(f"  level {lvl}  {lab:<28} n={k:>4}")
print(f"on_adv_support_baseline (level >= 1): {int(on_adv_support_baseline.sum())}  "
      f"[expected ~207]  (cohort-definition, not a covariate)")
print(f"  components -> HFNC {int(on_hfnc_baseline.sum())} | CPAP {int(on_cpap_baseline.sum())} | "
      f"IMV {int(on_imv_baseline.sum())} (incl. home)")
print(f"basec ({len(BASEC)} vars): {BASEC}")
miss = L0[BASEC].isna().mean().sort_values(ascending=False) * 100
print("\nbasec missingness (%):")
print(miss[miss > 0].round(1).to_string() if (miss > 0).any() else "  (none)")


L0: 1540 patients x 16 variables
home-invasive vent (medhx_homemechvent): 12  (folded into baseline IMV; 1 not otherwise flagged)
Baseline respiratory gradient (all randomised):
  level 0  none                         n=1333
  level 1  intermediate HFNC/NIV/CPAP   n=  97
  level 2  IMV (incl. home-invasive)    n= 110
on_adv_support_baseline (level >= 1): 207  [expected ~207]  (cohort-definition, not a covariate)
  components -> HFNC 78 | CPAP 63 | IMV 110 (incl. home)
basec (11 vars): ['age', 'female', 'sofa_baseline', 'lactate_baseline', 'creatinine_baseline', 'sf_ratio_baseline', 'map_baseline', 'ckd_any', 'sepsis_source_pulmonary', 'on_vasopressor_baseline', 'prerand_fluid_L']

basec missingness (%):
lactate_baseline       11.0
sf_ratio_baseline       4.7
creatinine_baseline     1.4
map_baseline            0.5


## B3. Treatment delivery

Fluid volume, vasopressor hours and new renal replacement therapy over the first 24 hours follow from the strategy. They are described but never adjusted for.

In [8]:
# Treatment delivery: described, never adjusted for
fluid_derived = derived_baseline.groupby('id').agg({
    'd_ipfluid6': 'first', 'd_ipfluid24': 'first', 'd_prerand_fltotal': 'first',
})
total_fluid_0_24h_mL = (fluid_derived['d_prerand_fltotal']
                        + fluid_derived['d_ipfluid6'] + fluid_derived['d_ipfluid24'])
vaso_hours_0_24h = derived_baseline.groupby('id')['d_totalvaso'].first()

# New RRT within 0-24h (excl. chronic dialysis). BioLINCC stripped rand_dt; use
# study_cloversenrolldt (= randomization date) with date-level resolution (<=1 day ~ 0-24h).
rrt_info = df_raw.groupby('id').agg({
    'rrthx_newyn': 'first', 'rrthx_firstdt': 'first', 'study_cloversenrolldt': 'first',
}).reindex(all_patient_ids)
enroll_dt   = pd.to_datetime(rrt_info['study_cloversenrolldt'], errors='coerce')
rrt_first   = pd.to_datetime(rrt_info['rrthx_firstdt'], errors='coerce')
days_to_rrt = (rrt_first - enroll_dt).dt.days
has_new_rrt = _is_yes(rrt_info['rrthx_newyn'])
rrt_early   = pd.Series(0, index=pd.Index(all_patient_ids, name='id'), dtype=int)
rrt_early.loc[has_new_rrt & days_to_rrt.between(0, 1)] = 1
n_rrt_nodate = int((has_new_rrt & rrt_first.isna()).sum())
if n_rrt_nodate:
    print(f"  {n_rrt_nodate} new-RRT patients missing firstdt -> coded 0 (may undercount)")

MECH = pd.DataFrame(index=pd.Index(all_patient_ids, name='id'))
MECH['fluid_0_24h_L']    = total_fluid_0_24h_mL / 1000
MECH['vaso_hours_0_24h'] = vaso_hours_0_24h
MECH['rrt_0_24h']        = rrt_early

print(f"MECH (delivery/mechanism): {len(MECH)} x {len(MECH.columns)}  "
      f"[NOT confounders; excluded from every adjustment set]")
_rrt_by_arm = MECH[['rrt_0_24h']].join(A[['restrictive']]).groupby('restrictive')['rrt_0_24h'].agg(['sum','mean'])
_rrt_by_arm.index = ['Liberal', 'Restrictive']
print(f"new RRT 0-24h: {int(MECH['rrt_0_24h'].sum())} total  [Liberal {int(_rrt_by_arm.loc['Liberal','sum'])} | "
      f"Restrictive {int(_rrt_by_arm.loc['Restrictive','sum'])}]  (expected ~45, balanced)")

MECH (delivery/mechanism): 1540 x 3  [NOT confounders; excluded from every adjustment set]
new RRT 0-24h: 48 total  [Liberal 24 | Restrictive 24]  (expected ~45, balanced)


## B4. Mediators

Renal: `delta_creat_0_24h`, creatinine at 24 hours minus creatinine at baseline. Pulmonary: `resp_escalation_24h`, an upward move in respiratory support within 24 hours, from level 0 to any advanced support or from level 1 to invasive ventilation. Alternative definitions (Results E2): new advanced support in patients without it at baseline (`new_adv_support_24h`), new invasive ventilation (`new_imv_24h`) and the change in SpO2/FiO2 (`delta_sf_ratio_0_24h`). The respiratory variables are rebuilt from the raw `ipvent_*` flags.

In [9]:
# Mediators
#   renal    : delta_creat_0_24h        (continuous, linear)
#   pulm     : resp_escalation_24h      (binary, logistic)  <- pulmonary mediator
#              strict upward move on {0 none, 1 intermediate, 2 IMV} in 0-24h, among patients
#              with headroom (baseline level 0 or 1). Level-0 -> any advanced support counts;
#              level-1 -> only IMV counts; baseline level-2 (IMV incl. home-invasive) -> NaN.
#   sens     : new_adv_support_24h      (binary, logistic)  <- de-novo escalation, level-0 only
#   nested   : new_imv_24h              (binary, logistic)  <- invasive only
#   sf       : delta_sf_ratio_0_24h     (continuous, linear) <- change in SpO2/FiO2

# renal + SF deltas
creat_24h    = day1.groupby('id')['labs_creatinine'].first()
oxy_24h      = day1.groupby('id').agg({'labs_spo2': 'first', 'labs_spo2fio2': 'first'})
sf_ratio_24h = (oxy_24h['labs_spo2'] / oxy_24h['labs_spo2fio2']).replace([np.inf, -np.inf], np.nan)
delta_creat    = creat_24h - labs_baseline['creatinine_baseline']
delta_sf_ratio = sf_ratio_24h - labs_baseline['sf_ratio_baseline']

# pulmonary escalation from raw ipvent_* (reuse sip_flag + baseline gradient from L0)
# peak respiratory level reached in 0-24h (0 none | 1 intermediate | 2 IMV)
_imv_win = sip_flag('ipvent_iventh6') | sip_flag('ipvent_iventh24')
_int_win = (sip_flag('ipvent_hfo2h6') | sip_flag('ipvent_hfo2h24') |
            sip_flag('ipvent_cpaph6') | sip_flag('ipvent_cpaph24'))
peak_resp_lvl = np.maximum(base_resp_lvl.values, np.where(_imv_win, 2, np.where(_int_win, 1, 0)))

# Pulmonary mediator: upward move, defined only for baseline level 0 or 1
# (NaN for level 2, where no escalation is possible, outside the escalation cohort).
resp_escalation_24h = pd.Series(
    np.where(base_resp_lvl.values <= 1, (peak_resp_lvl > base_resp_lvl.values).astype(float), np.nan),
    index=pd.Index(all_patient_ids, name='id'))

# alternative mediators: new advanced support (level 0 only) and new invasive ventilation
new_imv_24h = ((~on_imv_baseline) &
               (sip_flag('ipvent_iventh6') | sip_flag('ipvent_iventh24'))).astype(int)
new_adv_support_24h = ((~on_adv_support_baseline.astype(bool)) & (_imv_win | _int_win)).astype(int)

# assemble M
M = pd.DataFrame(index=pd.Index(all_patient_ids, name='id'))
M['creatinine_24h']         = creat_24h
M['delta_creat_0_24h']      = delta_creat
M['delta_sf_ratio_0_24h']   = delta_sf_ratio
M['resp_escalation_24h']    = resp_escalation_24h.reindex(M.index)
M['new_adv_support_24h']    = new_adv_support_24h.reindex(M.index)
M['new_imv_24h']            = new_imv_24h.reindex(M.index)

# validation: reconstruction vs derived vars, and gradient internal invariants
def _der_pp(col):
    if col not in df_derived.columns:
        return None
    return df_derived.groupby('id')[col].agg(
        lambda x: x.dropna().iloc[0] if x.notna().any() else np.nan).reindex(M.index)

print("Reconstruction cross-check (raw ipvent_* vs derived d_*):")
for recon_name, recon, dname in [('new_imv_24h', new_imv_24h, 'd_ivent24')]:
    dv = _der_pp(dname)
    if dv is not None:
        dv_yes = _is_yes(dv)
        agree = (recon.reindex(M.index).astype(int) == dv_yes.astype(int)).mean() * 100
        print(f"  {recon_name}: recon {int(recon.sum())} events, {dname} {int(dv_yes.sum())}, "
              f"agreement {agree:.1f}%")

# gradient invariants: on level-0, escalation == de-novo composite; on level-1, escalation == reached IMV
_lvl = base_resp_lvl.reindex(M.index)
_e0 = (M.loc[_lvl == 0, 'resp_escalation_24h'].fillna(-1).astype(int) ==
       M.loc[_lvl == 0, 'new_adv_support_24h']).mean() * 100
_reached_imv = (_imv_win.reindex(M.index)).astype(int)
_e1 = (M.loc[_lvl == 1, 'resp_escalation_24h'].astype('Int64') ==
       _reached_imv.loc[_lvl == 1]).mean() * 100
print(f"gradient invariants: level-0 escalation==de-novo composite {_e0:.1f}% | "
      f"level-1 escalation==reached-IMV {_e1:.1f}%  (both expected 100)")

print("\nMediator missingness (%):")
print((M.isna().mean() * 100).round(1).to_string())
print(f"\n>>> M_renal delta_creat_0_24h  : {(M['delta_creat_0_24h'].isna().mean()*100):.1f}% missing")
print(f">>> M_pulm  resp_escalation_24h: {(M['resp_escalation_24h'].isna().mean()*100):.1f}% missing "
      f"(NaN = baseline level-2, excluded by the headroom cohort)")

# incidence by arm (all randomised, before landmark/headroom)
_arm = A['restrictive'].reindex(M.index)
for flag, name in [('resp_escalation_24h', 'gradient escalation'),
                   ('new_adv_support_24h', 'de-novo advanced'),
                   ('new_imv_24h', 'new invasive')]:
    kL = int(M.loc[_arm == 0, flag].sum()); kR = int(M.loc[_arm == 1, flag].sum())
    print(f"  {name:<20}: Liberal {kL} | Restrictive {kR} | overall {int(M[flag].sum())}")


Reconstruction cross-check (raw ipvent_* vs derived d_*):
  new_imv_24h: recon 93 events, d_ivent24 94, agreement 99.9%
gradient invariants: level-0 escalation==de-novo composite 100.0% | level-1 escalation==reached-IMV 100.0%  (both expected 100)

Mediator missingness (%):
creatinine_24h           8.4
delta_creat_0_24h        9.7
delta_sf_ratio_0_24h    10.0
resp_escalation_24h      7.1
new_adv_support_24h      0.0
new_imv_24h              0.0

>>> M_renal delta_creat_0_24h  : 9.7% missing
>>> M_pulm  resp_escalation_24h: 7.1% missing (NaN = baseline level-2, excluded by the headroom cohort)
  gradient escalation : Liberal 89 | Restrictive 73 | overall 162
  de-novo advanced    : Liberal 81 | Restrictive 60 | overall 141
  new invasive        : Liberal 48 | Restrictive 45 | overall 93


## B5. Outcomes

In [10]:
# Y: outcomes
#   mortality_90d          (secondary)
#   alive_discharged_28d   (primary): discharged alive by day 28, not dead by day 28
#   alive_at_24h           (landmark gate)
Y = baseline.groupby('id')['term_d90stat'].first().to_frame()
Y['mortality_90d'] = (Y['term_d90stat'] == 'Dead').astype(int)

death_day = pd.to_numeric(baseline.groupby('id')['term_deathdt'].first(), errors='coerce')
alive_at_24h = pd.Series(1, index=Y.index, dtype=int)
alive_at_24h[death_day.notna() & (death_day <= 1)] = 0
Y['alive_at_24h'] = alive_at_24h

term_vars = baseline.groupby('id').agg({
    'term_disch': 'first', 'term_dischdt': 'first', 'term_deathdt': 'first',
})
disch_day = pd.to_numeric(term_vars['term_dischdt'], errors='coerce')
death_d   = pd.to_numeric(term_vars['term_deathdt'], errors='coerce')
alive_discharged_28d = pd.Series(0, index=term_vars.index)
alive_discharged_28d[(term_vars['term_disch'] == 'Yes') & (disch_day <= 28) &
                     (death_d.isna() | (death_d > 28))] = 1
alive_discharged_28d[term_vars['term_disch'].isna()] = np.nan
Y['alive_discharged_28d'] = alive_discharged_28d

print(f"90-day mortality      : {Y['mortality_90d'].mean()*100:.1f}% (n={int(Y['mortality_90d'].sum())})")
print(f"Alive at 24h          : {Y['alive_at_24h'].mean()*100:.1f}%")
print(f"28d alive-discharged  : {np.nanmean(Y['alive_discharged_28d'])*100:.1f}% "
      f"(missing {Y['alive_discharged_28d'].isna().mean()*100:.1f}%)")

90-day mortality      : 21.8% (n=336)
Alive at 24h          : 97.7%
28d alive-discharged  : 80.4% (missing 0.0%)


## B6. Analysis cohorts

Landmark cohort (alive at 24 hours, 1,504 patients), escalation cohort (landmark patients at level 0 or 1, 1,408) and the patients at level 0 (1,317).

In [11]:
# Merge, 24-hour landmark and cohorts
df_all = (L0.join(A[['restrictive']]).join(MECH).join(M)
            .join(Y[['alive_at_24h', 'mortality_90d', 'alive_discharged_28d']]))

# 24h landmark
n_before = len(df_all)
df_med = df_all[df_all['alive_at_24h'] == 1].drop(columns=['alive_at_24h']).copy()
print(f"24h landmark cohort : {len(df_med)} / {n_before}   [expected 1504]")

# respiratory-gradient cohorts
#   headroom = escalation cohort (baseline level 0 or 1): pulmonary and joint models
#   at_risk  = baseline level 0: alternative pulmonary mediator
df_med['headroom'] = (df_med['base_resp_lvl'] <= 1).astype(int)
df_med['at_risk']  = (df_med['base_resp_lvl'] == 0).astype(int)
n_head   = int(df_med['headroom'].sum())
n_atrisk = int(df_med['at_risk'].sum())
print("Baseline respiratory level (landmark):")
for lvl, lab in [(0,'none'), (1,'intermediate HFNC/NIV/CPAP'), (2,'IMV (incl. home-invasive)')]:
    k = int((df_med['base_resp_lvl'] == lvl).sum())
    print(f"  level {lvl}  {lab:<28} n={k:>4} ({k/len(df_med)*100:4.1f}%)")
print(f"headroom cohort (level 0 or 1): {n_head}   [pulmonary and joint models]")
print(f"at-risk cohort  (level 0)     : {n_atrisk}   [alternative pulmonary mediator]")

# event counts in the escalation cohort
hr = df_med[df_med['headroom'] == 1]
for flag, name in [('resp_escalation_24h', 'gradient escalation'), ('new_imv_24h', 'new invasive')]:
    kL = int(hr.loc[hr['restrictive'] == 0, flag].sum()); kR = int(hr.loc[hr['restrictive'] == 1, flag].sum())
    nL = int((hr['restrictive'] == 0).sum()); nR = int((hr['restrictive'] == 1).sum())
    print(f"  {name:<20} (headroom): Liberal {kL}/{nL} ({kL/nL*100:.1f}%) | "
          f"Restrictive {kR}/{nR} ({kR/nR*100:.1f}%) | total {kL+kR}")

# missingness and complete cases (joint model, escalation cohort)
print("\nHeadroom missingness (%) for basec + mediators + outcome:")
_cols = BASEC + [M_RENAL, M_PULM, PRIMARY_OUTCOME]
_cols = [c for c in _cols if c in hr.columns]
_miss = (hr[_cols].isna().mean() * 100).sort_values(ascending=False)
print(_miss[_miss > 0].round(1).to_string() if (_miss > 0).any() else "  (none)")
n_cc = hr[_cols].dropna().shape[0]
print(f"\nComplete cases (escalation cohort, joint model): {n_cc}/{n_head} "
      f"({n_cc/n_head*100:.1f}%)  ->  MI {'ON' if (hr[_cols].isna().mean().max()>0.05) else 'not needed'}")


24h landmark cohort : 1504 / 1540   [expected 1504]
Baseline respiratory level (landmark):
  level 0  none                         n=1317 (87.6%)
  level 1  intermediate HFNC/NIV/CPAP   n=  91 ( 6.1%)
  level 2  IMV (incl. home-invasive)    n=  96 ( 6.4%)
headroom cohort (level 0 or 1): 1408   [pulmonary and joint models]
at-risk cohort  (level 0)     : 1317   [alternative pulmonary mediator]
  gradient escalation  (headroom): Liberal 81/705 (11.5%) | Restrictive 66/703 (9.4%) | total 147
  new invasive         (headroom): Liberal 40/705 (5.7%) | Restrictive 38/703 (5.4%) | total 78

Headroom missingness (%) for basec + mediators + outcome:
lactate_baseline       10.9
delta_creat_0_24h       8.4
sf_ratio_baseline       4.6
creatinine_baseline     1.1
map_baseline            0.4

Complete cases (escalation cohort, joint model): 1105/1408 (78.5%)  ->  MI ON


# Part C. Description

Effect of the strategy on the pulmonary mediators, respiratory levels and their changes, treatment delivery by arm, and the renal cohorts of Results E1.

## C1. Effect of the strategy on the pulmonary mediators

Logistic regression of each pulmonary mediator on the strategy and baseline covariates. Covariates that would cause perfect separation are dropped.

In [12]:
# C1: effect of the strategy on the pulmonary mediators
from scipy import stats as _stats
try:
    import statsmodels.formula.api as smf
    HAVE_SM = True
except ImportError:
    HAVE_SM = False
    print("(statsmodels unavailable -> crude only)")

# Frames used in Parts C to E
df_landmark = df_med.copy()                                  # 24h-landmark, n=1504
df_headroom = df_med[df_med['headroom'] == 1].copy()         # escalation cohort (level 0 or 1)
df_atrisk   = df_med[df_med['at_risk'] == 1].copy()          # level 0 only

# covariates: basec, with the baseline respiratory level
_CONT   = ['age', 'sofa_baseline', 'creatinine_baseline', 'map_baseline', 'prerand_fluid_L']
_BINCAT = ['ckd_any', 'sepsis_source_pulmonary', 'on_vasopressor_baseline', 'on_intermediate_baseline']

def safe_confounders(d, outcome):
    keep = [c for c in _CONT if c in d.columns and d[c].nunique() > 1]
    for c in _BINCAT:
        if c not in d.columns or d[c].nunique() < 2:
            continue
        ct = pd.crosstab(d[c], d[outcome])
        if (ct.values.min() == 0) and (d[c].nunique() <= 6):
            continue
        keep.append(c)
    return keep

def am_gate(cohort, flag, label):
    print("=" * 74); print(label); print("=" * 74)
    sub = cohort.dropna(subset=[flag]).copy()
    lib = sub.loc[sub['restrictive'] == 0, flag]
    res = sub.loc[sub['restrictive'] == 1, flag]
    pL, pR = lib.mean(), res.mean()
    _, pf = _stats.fisher_exact(pd.crosstab(sub['restrictive'], sub[flag]))
    print(f"  n={len(sub)}")
    print(f"  Liberal      {int(lib.sum()):>4} / {len(lib):<4} ({pL*100:.1f}%)")
    print(f"  Restrictive  {int(res.sum()):>4} / {len(res):<4} ({pR*100:.1f}%)")
    print(f"  crude restrictive/liberal RR = {pR/pL:.2f}, Fisher p = {pf:.4f}")
    if HAVE_SM:
        conf = safe_confounders(sub, flag)
        d = sub[[flag, 'restrictive'] + conf].dropna()
        try:
            r = smf.logit(f"{flag} ~ restrictive + " + " + ".join(conf), data=d).fit(disp=0)
            orr = np.exp(r.params['restrictive']); ci = np.exp(r.conf_int().loc['restrictive'])
            print(f"  adjusted OR(restrictive vs liberal) = {orr:.2f} "
                  f"[{ci[0]:.2f}, {ci[1]:.2f}], p = {r.pvalues['restrictive']:.4f}   (n={len(d)})")
            print(f"    covariates kept: {conf}")
        except Exception as e:
            print(f"  adjusted fit failed ({type(e).__name__}); CMAverse uses a penalized fit downstream")
    print()

# Pulmonary mediator, escalation cohort
am_gate(df_headroom, 'resp_escalation_24h',
        "Escalation of respiratory support (escalation cohort)")
# New invasive ventilation, escalation cohort
am_gate(df_headroom, 'new_imv_24h',
        "New invasive ventilation (escalation cohort)")
# New advanced support, baseline level 0 only
am_gate(df_atrisk, 'new_adv_support_24h',
        "New advanced support (baseline level 0 only)")


Escalation of respiratory support (escalation cohort)
  n=1408
  Liberal        81 / 705  (11.5%)
  Restrictive    66 / 703  (9.4%)
  crude restrictive/liberal RR = 0.82, Fisher p = 0.2224
  adjusted OR(restrictive vs liberal) = 0.77 [0.54, 1.10], p = 0.1521   (n=1386)
    covariates kept: ['age', 'sofa_baseline', 'creatinine_baseline', 'map_baseline', 'prerand_fluid_L', 'ckd_any', 'sepsis_source_pulmonary', 'on_vasopressor_baseline', 'on_intermediate_baseline']

New invasive ventilation (escalation cohort)
  n=1408
  Liberal        40 / 705  (5.7%)
  Restrictive    38 / 703  (5.4%)
  crude restrictive/liberal RR = 0.95, Fisher p = 0.9074
  adjusted OR(restrictive vs liberal) = 0.91 [0.57, 1.47], p = 0.7116   (n=1386)
    covariates kept: ['age', 'sofa_baseline', 'creatinine_baseline', 'map_baseline', 'prerand_fluid_L', 'ckd_any', 'sepsis_source_pulmonary', 'on_vasopressor_baseline', 'on_intermediate_baseline']

New advanced support (baseline level 0 only)
  n=1317
  Liberal        75 

## C2. Respiratory levels and escalation events

Level at baseline, changes within 24 hours and the event count behind each pulmonary mediator. Patients on invasive ventilation at baseline cannot escalate and are left out. Patients on non-invasive support count an event only if they move to invasive ventilation.

In [13]:
# C2: respiratory levels, transitions, cohorts and event counts
# Baseline state is the gradient level built in the L0 cell (home-invasive folded into IMV).
state_b = df_landmark['base_resp_lvl'].astype(int).values

print("Baseline respiratory state (24h-landmark cohort, n=%d):" % len(df_landmark))
for s, lab in [(0, 'none / standard O2'), (1, 'intermediate (HFNC/NIV/CPAP)'), (2, 'IMV (incl. home-invasive)')]:
    n = int((state_b == s).sum())
    print(f"  state {s}  {lab:<30} n={n:>4}  ({n/len(df_landmark)*100:4.1f}%)")

# cohorts on the respiratory gradient
# headroom : baseline level 0 or 1 (escalation possible). The pulmonary mediator
#            counts an upward move: level 0 to any advanced support, level 1 to IMV.
# level 0  : new advanced support from none (alternative mediator).
# Baseline level 2 (IMV) cannot escalate and is excluded from every escalation model.
n_land   = len(df_landmark)
n_head   = int((state_b <= 1).sum())                       # levels 0 and 1
n_atrisk = int((state_b == 0).sum())                       # level 0 only
print("\nRespiratory-gradient cohorts (landmark):")
print(f"  all landmark survivors                          : {n_land}")
print(f"  headroom (level 0 or 1)  [pulmonary, joint]      : {n_head}")
print(f"  level-0 only             [alternative mediator]  : {n_atrisk}")
print(f"  excluded (level 2, IMV incl. home-invasive)      : {int((state_b == 2).sum())}")

# event reconciliation
resp_esc = df_landmark['resp_escalation_24h']               # NaN on level-2 (excluded)
new_imv  = df_landmark['new_imv_24h'].astype(int)
new_adv  = df_landmark['new_adv_support_24h'].astype(int)

esc_head        = int(resp_esc[state_b <= 1].sum())         # gradient events on headroom
esc_from_lvl1   = int(resp_esc[state_b == 1].sum())         # of those, contributed by level-1 -> IMV
denovo_lvl0     = int(new_adv[state_b == 0].sum())          # de-novo events (level 0)
imv_prog_events = int(new_imv[state_b != 2].sum())          # new IMV among levels 0 and 1
print("\nEscalation events:")
print(f"  gradient escalation, headroom (n={n_head})         : {esc_head}")
print(f"    of which contributed by level-1 -> IMV          : {esc_from_lvl1}")
print(f"  de-novo advanced, level-0 (n={n_atrisk})           : {denovo_lvl0}")
print(f"  new IMV among levels 0-1 (progression + de-novo)   : {imv_prog_events}")


Baseline respiratory state (24h-landmark cohort, n=1504):
  state 0  none / standard O2             n=1317  (87.6%)
  state 1  intermediate (HFNC/NIV/CPAP)   n=  91  ( 6.1%)
  state 2  IMV (incl. home-invasive)      n=  96  ( 6.4%)

Respiratory-gradient cohorts (landmark):
  all landmark survivors                          : 1504
  headroom (level 0 or 1)  [pulmonary, joint]      : 1408
  level-0 only             [alternative mediator]  : 1317
  excluded (level 2, IMV incl. home-invasive)      : 96

Escalation events:
  gradient escalation, headroom (n=1408)         : 147
    of which contributed by level-1 -> IMV          : 17
  de-novo advanced, level-0 (n=1317)           : 130
  new IMV among levels 0-1 (progression + de-novo)   : 78


## C3. Treatment delivery by arm

Fluid volume and vasopressor exposure differ between arms, as intended by the trial. New renal replacement therapy within 24 hours is balanced.

In [16]:
# C3: treatment delivery by arm
ARM = {0: 'Liberal', 1: 'Restrictive'}          # df_landmark['restrictive'] is integer 0/1
mech_cont = [c for c in ['fluid_0_24h_L', 'vaso_hours_0_24h'] if c in df_landmark.columns]
print("Mechanism variables by arm (24h-landmark cohort):")
print("-" * 70)
for c in mech_cont:
    d = df_landmark[[c, 'restrictive']].dropna()
    vL = d.loc[d['restrictive'] == 0, c]
    vR = d.loc[d['restrictive'] == 1, c]
    if len(vL) == 0 or len(vR) == 0:
        print(f"  {c:<18} (no non-missing values in one arm; column empty in this cohort?)")
        continue
    try:
        _, p = _stats.mannwhitneyu(vL, vR)
    except Exception:
        p = np.nan
    print(f"  {c:<18} median  Liberal={vL.median():.2f}  Restrictive={vR.median():.2f}   Mann-Whitney p={p:.2e}")

# Early RRT balance
rrt = df_landmark[['rrt_0_24h', 'restrictive']].dropna()
kL = int(rrt.loc[rrt['restrictive'] == 0, 'rrt_0_24h'].sum())
kR = int(rrt.loc[rrt['restrictive'] == 1, 'rrt_0_24h'].sum())
_, pr = _stats.fisher_exact(pd.crosstab(rrt['restrictive'], rrt['rrt_0_24h']))
print(f"\n  rrt_0_24h (incident RRT)   Liberal={kL}  Restrictive={kR}   Fisher p={pr:.3f}   -> balanced")


Mechanism variables by arm (24h-landmark cohort):
----------------------------------------------------------------------
  fluid_0_24h_L      median  Liberal=5.41  Restrictive=3.30   Mann-Whitney p=8.53e-103
  vaso_hours_0_24h   median  Liberal=0.00  Restrictive=5.07   Mann-Whitney p=8.89e-20

  rrt_0_24h (incident RRT)   Liberal=22  Restrictive=23   Fisher p=0.881   -> balanced


## C4. Renal mediator and the renal cohorts

Creatinine does not reflect the kidney response in patients on chronic dialysis or starting renal replacement therapy within 24 hours. Results E1 repeats the renal model without them.

In [17]:
# C4: renal mediator and the renal cohorts of Results E1
ARM = {0: 'Liberal', 1: 'Restrictive'}          # df_landmark['restrictive'] is integer 0/1
dc = df_landmark[['delta_creat_0_24h', 'restrictive']].dropna()
for code, lab in ARM.items():
    v = dc.loc[dc['restrictive'] == code, 'delta_creat_0_24h']
    if len(v):
        print(f"  Δcreatinine 0-24h  {lab:<12} median={v.median():+.3f}  mean={v.mean():+.3f}  n={len(v)}")
    else:
        print(f"  Δcreatinine 0-24h  {lab:<12} (no non-missing values)")
try:
    _, pdc = _stats.mannwhitneyu(dc.loc[dc['restrictive'] == 0, 'delta_creat_0_24h'],
                                 dc.loc[dc['restrictive'] == 1, 'delta_creat_0_24h'])
    print(f"  Mann-Whitney (Δcreat by arm): p={pdc:.3f}")
except Exception:
    pass

n_chronic = int((df_landmark['chronic_dialysis'] == 1).sum())
n_rrt024  = int((df_landmark['rrt_0_24h'] == 1).sum())
n_either  = int(((df_landmark['chronic_dialysis'] == 1) | (df_landmark['rrt_0_24h'] == 1)).sum())
print("\nRenal cleaning cohorts (from the full landmark, n=%d):" % len(df_landmark))
print(f"  baseline chronic dialysis          : {n_chronic}")
print(f"  incident RRT 0-24h                 : {n_rrt024}")
print(f"  either (excluded in combined check): {n_either}")
print(f"  -> renal-clean cohort n = {len(df_landmark) - n_either}")
if n_chronic == 0:
    print("  (baseline chronic dialysis is 0 here; the exclusion is a documented no-op, kept for symmetry.)")
else:
    print(f"  ({n_chronic} patients on chronic dialysis are in the landmark cohort. Their creatinine")
    print("   follows the dialysis schedule rather than the kidney response.)")


  Δcreatinine 0-24h  Liberal      median=-0.210  mean=-0.366  n=692
  Δcreatinine 0-24h  Restrictive  median=-0.140  mean=-0.238  n=686
  Mann-Whitney (Δcreat by arm): p=0.000

Renal cleaning cohorts (from the full landmark, n=1504):
  baseline chronic dialysis          : 76
  incident RRT 0-24h                 : 45
  either (excluded in combined check): 121
  -> renal-clean cohort n = 1383
  (76 patients on chronic dialysis are in the landmark cohort. Their creatinine
   follows the dialysis schedule rather than the kidney response.)


# Part D. CMAverse g-formula

Fourteen models, each with a strategy-by-mediator interaction and a logistic outcome model.

- Model 1: joint model with both mediators, escalation cohort.
- Model 2: renal pathway alone, landmark cohort.
- Model 3: pulmonary pathway alone, escalation cohort.
- Model 4: renal pathway alone, escalation cohort.
- Models 5 to 7: models 1 to 3 on 90-day mortality.
- Models 8 to 10: alternative pulmonary mediators.
- Models 11 to 14: renal pathway with non-invasive or invasive support at baseline added to the adjustment set, without chronic dialysis, without early renal replacement therapy, and without both.

No post-randomisation variable is adjusted for (`postc` is empty).

## D1. Export the analysis cohorts

Writes `cohort_landmark.csv` and `cohort_headroom.csv`, patient-level files kept in `data/mediation`.

In [18]:
# D1: export the escalation (headroom, n = 1408) and landmark (n = 1504) cohorts
_export_cols = ['restrictive'] + [c for c in BASEC] + [
    'on_adv_support_baseline', 'on_mech_vent_baseline', 'on_intermediate_baseline',
    'base_resp_lvl', 'chronic_dialysis', 'rrt_0_24h',
    M_RENAL, M_PULM, M_PULM_COMPOSITE, M_PULM_SENS_SF, M_PULM_NESTED,
    PRIMARY_OUTCOME, SECONDARY_OUTCOME,
]
def _dedup(seq):
    seen = set(); out = []
    for x in seq:
        if x not in seen and x is not None:
            seen.add(x); out.append(x)
    return out
_export_cols = _dedup(_export_cols)

csv_headroom = os.path.join(OUTPUT_DIR, 'cohort_headroom.csv')
csv_landmark = os.path.join(OUTPUT_DIR, 'cohort_landmark.csv')

_hr = df_headroom[[c for c in _export_cols if c in df_headroom.columns]].copy()
_lm = df_landmark[[c for c in _export_cols if c in df_landmark.columns]].copy()
_hr.to_csv(csv_headroom, index=False)
_lm.to_csv(csv_landmark, index=False)

print(f"headroom cohort : {_hr.shape[0]} x {_hr.shape[1]}  -> {csv_headroom}")
print(f"landmark cohort : {_lm.shape[0]} x {_lm.shape[1]}  -> {csv_landmark}")

# basec columns constant in the escalation cohort (R drops them with a note)
_const = [c for c in BASEC if c in _hr.columns and _hr[c].nunique(dropna=True) < 2]
print(f"\nBASEC columns constant within headroom (will be pruned by R): {_const if _const else 'none'}")
print("Event/outcome availability:")
for nm, fr in [('headroom', _hr), ('landmark', _lm)]:
    print(f"  {nm:<9} {M_PULM}={int(fr[M_PULM].sum())}  {M_PULM_COMPOSITE}={int(fr[M_PULM_COMPOSITE].sum())}  "
          f"{M_PULM_NESTED}={int(fr[M_PULM_NESTED].sum())}  {PRIMARY_OUTCOME} mean={fr[PRIMARY_OUTCOME].mean():.3f}")


headroom cohort : 1408 x 25  -> data\mediation\cohort_headroom.csv
landmark cohort : 1504 x 25  -> data\mediation\cohort_landmark.csv

BASEC columns constant within headroom (will be pruned by R): none
Event/outcome availability:
  headroom  resp_escalation_24h=147  new_adv_support_24h=130  new_imv_24h=78  alive_discharged_28d mean=0.835
  landmark  resp_escalation_24h=147  new_adv_support_24h=130  new_imv_24h=78  alive_discharged_28d mean=0.823


## D2. Write the CMAverse script

The settings are filled into an R template written to `data/mediation/run_cmaverse.R`.

In [19]:
# D2: write the CMAverse R script (placeholders replaced by the settings)
basec_r      = 'c(' + ', '.join('"%s"' % c for c in BASEC) + ')'
basec_pulm_r = 'c(' + ', '.join('"%s"' % c for c in (list(BASEC) + [PULM_COVAR])) + ')'
basec_rob_r  = 'c(' + ', '.join('"%s"' % c for c in (list(BASEC) + ['on_adv_support_baseline'])) + ')'

r_template = r'''#!/usr/bin/env Rscript
# CLOVERS 24-hour landmark mediation: CMAverse g-formula, natural direct and indirect effects
# Joint two-mediator model, single-mediator models and alternative analyses.
# Pulmonary mediator: escalation of respiratory support, escalation cohort (headroom, level 0 or 1),
#   adjusting for on_intermediate_baseline (BASEC_PULM).
# postc is empty: no post-randomisation confounder is adjusted for.

suppressPackageStartupMessages({
  if (!requireNamespace("CMAverse", quietly = TRUE)) {
    if (!requireNamespace("devtools", quietly = TRUE))
      install.packages("devtools", repos = "https://cloud.r-project.org")
    devtools::install_github("BS1125/CMAverse", upgrade = "never")
  }
  if (!requireNamespace("jsonlite", quietly = TRUE))
    install.packages("jsonlite", repos = "https://cloud.r-project.org")
  library(CMAverse); library(jsonlite)
})
cat(sprintf("CMAverse %s\n", packageVersion("CMAverse")))

set.seed(@@SEED@@)
NBOOT <- @@NBOOT@@L
NREP  <- @@NREP@@L    # passed to cmest, not used by the g-formula estimator
NMICE <- @@NMICE@@L    # mice imputations for missing data

df_headroom <- read.csv("@@CSV_HEADROOM@@", stringsAsFactors = FALSE)
df_landmark <- read.csv("@@CSV_LANDMARK@@", stringsAsFactors = FALSE)
df_headroom$restrictive <- as.integer(df_headroom$restrictive)
df_landmark$restrictive <- as.integer(df_landmark$restrictive)
cat(sprintf("headroom n=%d | landmark n=%d\n", nrow(df_headroom), nrow(df_landmark)))

BASEC      <- @@BASEC@@
BASEC_PULM <- @@BASEC_PULM@@
BASEC_ROB  <- @@BASEC_ROB@@

# ------------------------------------------------------------------ helpers
extract_effects <- function(res) {
  pe <- res$effect.pe; cl <- res$effect.ci.low; ch <- res$effect.ci.high; pv <- res$effect.pval
  if (is.null(pe) || length(pe) == 0) return(list())
  eff <- list()
  for (en in names(pe)) {
    eff[[en]] <- list(estimate = as.numeric(pe[[en]]),
                      ci_lower = as.numeric(cl[[en]]),
                      ci_upper = as.numeric(ch[[en]]),
                      p_value  = if (!is.null(pv) && en %in% names(pv)) as.numeric(pv[[en]]) else NA_real_)
  }
  eff
}

extract_evalues <- function(res) {
  tryCatch({
    sens <- tryCatch(cmsens(object = res, sens = "uc"),
                     error = function(e1) tryCatch(cmsens(object = res, sens.type = "uc"),
                                                   error = function(e2) NULL))
    if (is.null(sens) || is.null(sens$evalues)) return(list())
    ev_tbl <- sens$evalues
    evcols <- grep("^Evalue", colnames(ev_tbl), value = TRUE)
    ev_est_col <- evcols[grep("est",   evcols)[1]]
    ev_ci_col  <- evcols[grep("lower", evcols)[1]]
    if (is.na(ev_est_col) && ncol(ev_tbl) >= 4L) ev_est_col <- colnames(ev_tbl)[4]
    if (is.na(ev_ci_col)  && ncol(ev_tbl) >= 5L) ev_ci_col  <- colnames(ev_tbl)[5]
    evals <- list()
    for (rn in rownames(ev_tbl)) {
      est <- if (!is.na(ev_est_col)) as.numeric(ev_tbl[rn, ev_est_col]) else NA_real_
      ci  <- if (!is.na(ev_ci_col))  as.numeric(ev_tbl[rn, ev_ci_col])  else NA_real_
      if (!is.na(est)) evals[[rn]] <- list(evalue_est = est, evalue_ci = ci)
    }
    evals
  }, error = function(e) { cat(sprintf("    E-value error: %s\n", e$message)); list() })
}

RESULTS <- list()

run_cmest <- function(df, mediators, mregs, outcome, yreg, basec_use, label) {
  cat("\n"); cat(strrep("=", 70)); cat("\n"); cat(label); cat("\n"); cat(strrep("=", 70)); cat("\n")
  if (!all(c(mediators, outcome) %in% colnames(df))) { cat("  SKIP: missing columns\n"); return(invisible(NULL)) }
  basec_use <- basec_use[basec_use %in% colnames(df)]
  keep <- basec_use[sapply(basec_use, function(cc) length(unique(df[[cc]][!is.na(df[[cc]])])) > 1)]
  dropped <- setdiff(basec_use, keep)
  if (length(dropped) > 0) cat(sprintf("  pruned constant basec: %s\n", paste(dropped, collapse = ", ")))
  mcols <- c("restrictive", keep, mediators, outcome); mcols <- mcols[mcols %in% colnames(df)]
  max_miss <- max(colMeans(is.na(df[, mcols, drop = FALSE])))
  use_mi <- max_miss > 0.05
  cat(sprintf("  n=%d  max missing=%.1f%%  MI=%s (mice m=%d)  nRep=%d  nboot=%d\n",
              nrow(df), max_miss * 100, ifelse(use_mi, "YES", "NO"),
              ifelse(use_mi, NMICE, 0L), NREP, NBOOT))
  mval <- lapply(seq_along(mediators), function(i) {
    if (mregs[[i]] == "logistic") 0L else as.numeric(median(df[[mediators[i]]], na.rm = TRUE))
  })
  args <- list(data = df, model = "gformula", outcome = outcome, exposure = "restrictive",
               mediator = mediators, basec = keep, EMint = TRUE, mreg = mregs, yreg = yreg,
               astar = 0L, a = 1L, mval = mval,
               estimation = "imputation", inference = "bootstrap", nboot = NBOOT, nRep = NREP)
  if (use_mi) { args$multimp <- TRUE; args$args_mice <- list(m = NMICE) }
  res <- tryCatch(do.call(cmest, args),
                  error = function(e) { cat(sprintf("  FAILED: %s\n", e$message)); NULL })
  if (is.null(res)) { RESULTS[[length(RESULTS) + 1]] <<- list(label = label, error = TRUE); return(invisible(NULL)) }
  eff <- extract_effects(res); ev <- extract_evalues(res)
  for (en in grep("^R", names(eff), value = TRUE)) { e <- eff[[en]]   # ratio effects only
    cat(sprintf("  %-12s %.4f (%.4f, %.4f) p=%.4f\n",
                en, e$estimate, e$ci_lower, e$ci_upper, ifelse(is.na(e$p_value), NA, e$p_value))) }
  RESULTS[[length(RESULTS) + 1]] <<- list(label = label, mediators = as.list(mediators),
      outcome = outcome, yreg = yreg, n = nrow(res$data), basec = as.list(keep),
      effects = eff, evalues = ev)
  invisible(res)
}

# Primary outcome
# Joint two-mediator natural-effects model (headroom): PNDE + joint indirect
run_cmest(df_headroom, c("@@M_RENAL@@", "@@M_PULM@@"), list("linear", "logistic"),
          "@@Y_PRIMARY@@", "logistic", BASEC_PULM,
          "JOINT two-mediator | 28d alive-discharged [joint: PNDE + joint indirect]")

# Single renal (full landmark): indirect via renal
run_cmest(df_landmark, c("@@M_RENAL@@"), list("linear"),
          "@@Y_PRIMARY@@", "logistic", BASEC,
          "SINGLE renal (delta-creatinine) | 28d alive-discharged [indirect-via-renal]")

# Single pulmonary (headroom, gradient escalation): indirect via pulmonary
run_cmest(df_headroom, c("@@M_PULM@@"), list("logistic"),
          "@@Y_PRIMARY@@", "logistic", BASEC_PULM,
          "SINGLE pulmonary (gradient escalation) | 28d alive-discharged [indirect-via-pulmonary]")

# Single renal on the headroom cohort (compared with the joint model in Part E)
run_cmest(df_headroom, c("@@M_RENAL@@"), list("linear"),
          "@@Y_PRIMARY@@", "logistic", BASEC_PULM,
          "SINGLE renal (headroom) | 28d alive-discharged [overlap-cohort]")

# Secondary outcome (90-day mortality)
run_cmest(df_headroom, c("@@M_RENAL@@", "@@M_PULM@@"), list("linear", "logistic"),
          "@@Y_SECONDARY@@", "logistic", BASEC_PULM,
          "JOINT two-mediator | 90d mortality [secondary]")
run_cmest(df_landmark, c("@@M_RENAL@@"), list("linear"),
          "@@Y_SECONDARY@@", "logistic", BASEC,
          "SINGLE renal | 90d mortality [secondary]")
run_cmest(df_headroom, c("@@M_PULM@@"), list("logistic"),
          "@@Y_SECONDARY@@", "logistic", BASEC_PULM,
          "SINGLE pulmonary | 90d mortality [secondary]")

# Alternative analyses (Results E1 and E2)
# Pulmonary: new advanced support among patients at level 0 at baseline
run_cmest(subset(df_headroom, base_resp_lvl == 0), c("@@M_PULM_COMPOSITE@@"), list("logistic"),
          "@@Y_PRIMARY@@", "logistic", BASEC,
          "ALT pulmonary de-novo composite (level-0 only) | 28d alive-discharged")
# Pulmonary: IMV-only nested rung, headroom cohort
run_cmest(df_headroom, c("@@M_IMV@@"), list("logistic"),
          "@@Y_PRIMARY@@", "logistic", BASEC_PULM,
          "ALT pulmonary IMV-only (nested rung) | 28d alive-discharged")
# Pulmonary: change in SpO2/FiO2 as mediator
run_cmest(df_headroom, c("@@M_SF@@"), list("linear"),
          "@@Y_PRIMARY@@", "logistic", BASEC_PULM,
          "ALT pulmonary delta-SpO2/FiO2 (ceiling) | 28d alive-discharged")
# Renal: on_adv_support_baseline added to basec (full landmark)
run_cmest(df_landmark, c("@@M_RENAL@@"), list("linear"),
          "@@Y_PRIMARY@@", "logistic", BASEC_ROB,
          "ALT renal + on_adv_support_baseline in basec | 28d alive-discharged")
# Renal pathway without chronic dialysis, without early renal replacement therapy, or without both
run_cmest(subset(df_landmark, chronic_dialysis == 0), c("@@M_RENAL@@"), list("linear"),
          "@@Y_PRIMARY@@", "logistic", BASEC,
          "ALT renal excl. baseline dialysis | 28d alive-discharged")
run_cmest(subset(df_landmark, rrt_0_24h == 0), c("@@M_RENAL@@"), list("linear"),
          "@@Y_PRIMARY@@", "logistic", BASEC,
          "ALT renal excl. incident RRT 0-24h | 28d alive-discharged")
run_cmest(subset(df_landmark, chronic_dialysis == 0 & rrt_0_24h == 0), c("@@M_RENAL@@"), list("linear"),
          "@@Y_PRIMARY@@", "logistic", BASEC,
          "ALT renal excl. baseline dialysis AND incident RRT | 28d alive-discharged")

# Save
writeLines(toJSON(RESULTS, auto_unbox = TRUE, pretty = TRUE), "@@RESULTS_JSON@@")
cat(sprintf("\nSaved the model results: %s\n", "@@RESULTS_JSON@@"))

# Device 2 belongs to the rpy2 R magic, which closes it. Close any other device.
while (dev.cur() > 2L) dev.off()
'''

_repl = {
    '@@SEED@@': str(SEED),
    '@@NBOOT@@': str(N_BOOTSTRAP),
    '@@NREP@@': str(N_MC_REP),
    '@@NMICE@@': str(N_MICE),
    '@@CSV_HEADROOM@@': csv_headroom.replace(os.sep, '/'),
    '@@CSV_LANDMARK@@': csv_landmark.replace(os.sep, '/'),
    '@@BASEC@@': basec_r,
    '@@BASEC_PULM@@': basec_pulm_r,
    '@@BASEC_ROB@@': basec_rob_r,
    '@@M_RENAL@@': M_RENAL,
    '@@M_PULM@@': M_PULM,
    '@@M_PULM_COMPOSITE@@': M_PULM_COMPOSITE,
    '@@M_SF@@': M_PULM_SENS_SF,
    '@@M_IMV@@': M_PULM_NESTED,
    '@@Y_PRIMARY@@': PRIMARY_OUTCOME,
    '@@Y_SECONDARY@@': SECONDARY_OUTCOME,
    '@@RESULTS_JSON@@': os.path.join(OUTPUT_DIR, 'cmaverse_results.json').replace(os.sep, '/'),
}
r_script = r_template
for k, v in _repl.items():
    r_script = r_script.replace(k, v)

r_script_path = os.path.join(OUTPUT_DIR, 'run_cmaverse.R')
with open(r_script_path, 'w', encoding='utf-8') as f:
    f.write(r_script)

print(f"R script written: {r_script_path}")
print("Models: joint + renal (landmark) + pulmonary (escalation cohort) + renal (escalation cohort)")
print("        + 3 on 90-day mortality + 3 alternative pulmonary mediators + 4 alternative renal analyses")
print("Pulmonary and joint models on the escalation cohort with BASEC_PULM (adds on_intermediate_baseline).")
print("postc: EMPTY (omitted from every cmest call)")
print(f"nboot={N_BOOTSTRAP}, nRep={N_MC_REP}, mice m={N_MICE}")


R script written: data\mediation\run_cmaverse.R
Models: joint + renal (landmark) + pulmonary (escalation cohort) + renal (escalation cohort)
        + 3 on 90-day mortality + 3 alternative pulmonary mediators + 4 alternative renal analyses
Pulmonary and joint models on the escalation cohort with BASEC_PULM (adds on_intermediate_baseline).
postc: EMPTY (omitted from every cmest call)
nboot=1000, nRep=100, mice m=10


## D3. Run CMAverse

The long step, about 20 hours. The results go to `cmaverse_results.json`.

In [20]:
# D3: run the CMAverse script (long)
r_run_path = r_script_path.replace(os.sep, '/')
print("Sourcing:", r_run_path)


Sourcing: data/mediation/run_cmaverse.R


In [21]:
%%R -i r_run_path
# mice warnings ("Number of logged events") and E-value messages are not printed
suppressMessages(suppressWarnings(source(r_run_path, echo = FALSE)))


CMAverse 0.1.0
headroom n=1408 | landmark n=1504

JOINT two-mediator | 28d alive-discharged [joint: PNDE + joint indirect]
  n=1408  max missing=10.9%  MI=YES (mice m=10)  nRep=100  nboot=1000
  Rcde         1.2547 (0.9263, 1.7412) p=0.1440
  Rpnde        1.1252 (0.8681, 1.4756) p=0.3720
  Rtnde        1.1364 (0.8727, 1.4929) p=0.3260
  Rpnie        0.9859 (0.9296, 1.0390) p=0.6060
  Rtnie        0.9957 (0.9106, 1.0888) p=0.9240
  Rte          1.1203 (0.8628, 1.4667) p=0.3680

SINGLE renal (delta-creatinine) | 28d alive-discharged [indirect-via-renal]
  n=1504  max missing=11.2%  MI=YES (mice m=10)  nRep=100  nboot=1000
  Rcde         1.1199 (0.8600, 1.4316) p=0.3680
  Rpnde        1.1361 (0.8678, 1.4387) p=0.3260
  Rtnde        1.1159 (0.8549, 1.4304) p=0.3940
  Rpnie        0.9705 (0.9350, 0.9952) p=0.0120
  Rtnie        0.9532 (0.9064, 0.9902) p=0.0060
  Rte          1.0830 (0.8351, 1.3817) p=0.5280

SINGLE pulmonary (gradient escalation) | 28d alive-discharged [indirect-via-pulmona

# Part E. Results read from the CMAverse output

The effects below are the odds ratios returned by CMAverse. The paper gives risk ratios instead, computed from the same draws by `effect_scales.R`. Two results of this part are used in the paper as they are: the correlation between the two mediators (Table E3) and the Bonferroni-adjusted p-values of the two indirect effects, which are the same on both scales.

## E1. Load the results

In [22]:
# E1: load the CMAverse results into one table
import json

RESULTS_JSON = os.path.join(OUTPUT_DIR, 'cmaverse_results.json')
with open(RESULTS_JSON, encoding='utf-8') as f:
    _raw = json.load(f)

_rows = []
for _m in _raw:
    if _m.get('error'):
        _rows.append(dict(label=_m.get('label'), effect=None, n=np.nan))
        continue
    _ev = _m.get('evalues', {}) or {}
    for _en, _e in (_m.get('effects', {}) or {}).items():
        _e2 = _ev.get(_en, {}) or {}
        _rows.append(dict(
            label=_m['label'], outcome=_m.get('outcome'), n=_m.get('n'),
            mediators=tuple(_m.get('mediators', [])), effect=_en,
            estimate=_e.get('estimate'), lo=_e.get('ci_lower'),
            hi=_e.get('ci_upper'), p=_e.get('p_value'),
            ev_est=_e2.get('evalue_est', np.nan), ev_ci=_e2.get('evalue_ci', np.nan)))
RES = pd.DataFrame(_rows)
for _c in ('ev_est', 'ev_ci'):          # undefined E-values arrive as the string "NA"
    RES[_c] = pd.to_numeric(RES[_c], errors='coerce')

def geteff(label_tag, effect):
    """Return (estimate, lo, hi, p) for one effect of the model whose label contains label_tag."""
    sub = RES[(RES['label'].str.contains(label_tag, regex=False)) & (RES['effect'] == effect)]
    if sub.empty:
        raise KeyError(f'{label_tag!r} / {effect!r} not found in results')
    r = sub.iloc[0]
    return float(r['estimate']), float(r['lo']), float(r['hi']), float(r['p'])

print(f"Models in {os.path.basename(RESULTS_JSON)}:\n")
for lab in RES['label'].dropna().unique():
    _n = RES.loc[RES['label'] == lab, 'n'].iloc[0]
    print(f"  n={int(_n) if pd.notna(_n) else '   ?'}  {lab}")

Models in cmaverse_results.json:

  n=1408  JOINT two-mediator | 28d alive-discharged [joint: PNDE + joint indirect]
  n=1504  SINGLE renal (delta-creatinine) | 28d alive-discharged [indirect-via-renal]
  n=1408  SINGLE pulmonary (gradient escalation) | 28d alive-discharged [indirect-via-pulmonary]
  n=1408  SINGLE renal (headroom) | 28d alive-discharged [overlap-cohort]
  n=1408  JOINT two-mediator | 90d mortality [secondary]
  n=1504  SINGLE renal | 90d mortality [secondary]
  n=1408  SINGLE pulmonary | 90d mortality [secondary]
  n=1317  ALT pulmonary de-novo composite (level-0 only) | 28d alive-discharged
  n=1408  ALT pulmonary IMV-only (nested rung) | 28d alive-discharged
  n=1408  ALT pulmonary delta-SpO2/FiO2 (ceiling) | 28d alive-discharged
  n=1504  ALT renal + on_adv_support_baseline in basec | 28d alive-discharged
  n=1428  ALT renal excl. baseline dialysis | 28d alive-discharged
  n=1459  ALT renal excl. incident RRT 0-24h | 28d alive-discharged
  n=1383  ALT renal excl. b

## E2. Effects at the primary outcome

In [23]:
# E2: effects at the primary outcome (odds ratios, see the note above)
PNDE      = geteff('[joint: PNDE',              'Rpnde')  # pure natural direct effect (joint)
JOINT_IND = geteff('[joint: PNDE',              'Rtnie')  # joint natural indirect (both mediators)
IND_REN   = geteff('[indirect-via-renal]',      'Rtnie')  # single-renal total natural indirect
IND_PUL   = geteff('[indirect-via-pulmonary]',  'Rtnie')  # single-pulmonary total natural indirect

_cross = pd.DataFrame([
    ('Pure natural direct effect (PNDE)',   'joint two-mediator', 'rRpnde', *PNDE),
    ('Natural indirect via renal',          'single renal',       'rRtnie', *IND_REN),
    ('Natural indirect via pulmonary',      'single pulmonary',   'rRtnie', *IND_PUL),
    ('Joint natural indirect (both)',       'joint two-mediator', 'rRtnie', *JOINT_IND),
], columns=['estimand', 'model', 'effect', 'OR', 'lo', 'hi', 'p'])

pd.set_option('display.float_format', lambda v: f'{v:.4f}')
print("PRIMARY outcome: 28-day alive-and-discharged | contrast: restrictive vs liberal")
print("Ratio effects returned by CMAverse are odds ratios of the counterfactual probabilities.\n")
print(_cross.to_string(index=False))

PRIMARY outcome: 28-day alive-and-discharged | contrast: restrictive vs liberal
Ratio effects returned by CMAverse are odds ratios of the counterfactual probabilities.

                         estimand              model effect     OR     lo     hi      p
Pure natural direct effect (PNDE) joint two-mediator rRpnde 1.1252 0.8681 1.4756 0.3720
       Natural indirect via renal       single renal rRtnie 0.9532 0.9064 0.9902 0.0060
   Natural indirect via pulmonary   single pulmonary rRtnie 1.0388 0.9755 1.1145 0.2460
    Joint natural indirect (both) joint two-mediator rRtnie 0.9957 0.9106 1.0888 0.9240


## E3. Relationship between the two mediators

Point-biserial correlation between the creatinine change and escalation of respiratory support in the escalation cohort (Table E3). The excess relative risks printed below are on the odds-ratio scale, whereas Table E3 gives them on the risk-ratio scale.

In [24]:
# E3: correlation between the two mediators, and additivity of the two indirect effects
from scipy.stats import pointbiserialr

# escalation cohort = the rows used by the joint, pulmonary and escalation-cohort renal models
_hd = pd.read_csv(csv_headroom)

_pair = _hd[[M_RENAL, M_PULM]].dropna()
_r_pb, _p_pb = pointbiserialr(_pair[M_PULM].astype(int).values, _pair[M_RENAL].values)
print("Association between the two mediators\n")
print(f"Point-biserial r  (binary gradient escalation vs continuous Δcreatinine), "
      f"headroom n={len(_pair)}:")
print(f"  r = {_r_pb:+.3f}   p = {_p_pb:.3f}   shared variance r^2 = {_r_pb**2:.4f}\n")

# renal model on the same (escalation) cohort as the joint and pulmonary models, so that all three
# inputs come from one population
_ren = geteff('[overlap-cohort]', 'Rtnie'); _ren_src = 'headroom (n≈1408)'

_er_ren, _er_pul, _er_joint = _ren[0] - 1, IND_PUL[0] - 1, JOINT_IND[0] - 1
_sum_singles = _er_ren + _er_pul
_overlap_ER  = _er_joint - _sum_singles                     # = joint - renal - pulm + 1
print(f"Excess-relative-risk decomposition (rRtnie - 1), joint and both singles on the headroom")
print(f"cohort (renal single taken on {_ren_src}):")
print(f"  renal single        {_er_ren:+.4f}")
print(f"  pulmonary single    {_er_pul:+.4f}")
print(f"  sum of singles      {_sum_singles:+.4f}")
print(f"  joint indirect      {_er_joint:+.4f}")
print(f"  overlap (residual)  {_overlap_ER:+.4f}   [joint - renal - pulm + 1]")


Association between the two mediators

Point-biserial r  (binary gradient escalation vs continuous Δcreatinine), headroom n=1290:
  r = +0.003   p = 0.909   shared variance r^2 = 0.0000

Excess-relative-risk decomposition (rRtnie - 1), joint and both singles on the headroom
cohort (renal single taken on headroom (n≈1408)):
  renal single        -0.0391
  pulmonary single    +0.0388
  sum of singles      -0.0003
  joint indirect      -0.0043
  overlap (residual)  -0.0040   [joint - renal - pulm + 1]


## E4. Bonferroni correction

Over the two pathway-specific indirect effects at the primary outcome.

In [26]:
# E4: Bonferroni correction over the two indirect effects at the primary outcome
FAMILY = {
    'indirect via renal (rRtnie)':     IND_REN[3],
    'indirect via pulmonary (rRtnie)': IND_PUL[3],
}
_m, _alpha = len(FAMILY), 0.05
print("Family: the two pathway-specific total indirect effects (rRtnie) at the")
print(f"primary outcome, m={_m}.  Bonferroni threshold = {_alpha}/{_m} = {_alpha/_m:.4f}")
print("90-day mortality, a secondary outcome, is outside the family.\n")
for _name, _p in FAMILY.items():
    _padj = min(_p * _m, 1.0)
    _verdict = "remains significant" if _p < _alpha/_m else "not significant"
    print(f"  {_name:<34} p={_p:.4f}   adjusted p={_padj:.4f}   -> {_verdict} after correction")

Family: the two pathway-specific total indirect effects (rRtnie) at the
primary outcome, m=2.  Bonferroni threshold = 0.05/2 = 0.0250
90-day mortality, a secondary outcome, is outside the family.

  indirect via renal (rRtnie)        p=0.0060   adjusted p=0.0120   -> remains significant after correction
  indirect via pulmonary (rRtnie)    p=0.2460   adjusted p=0.4920   -> not significant after correction
